# ChessHack — Colab GPU trainer

**Division of labor (important):** Stockfish *labeling* is CPU-bound — run it on your 16-core box (`python -m trainer.label`). *Training* is GPU-bound — run it here on the 80GB GPU. Labeled shards travel local → Google Drive → here.

**Before running:** push this repo to GitHub and set `REPO_URL` below (or upload the repo folder to Drive and skip the clone).

Phase 1 (distillation) targets ~2000 Elo. Phase 2 (self-play, uncapped) is launched once Phase 1 lands.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/chesshack'
for d in ('data', 'data/distill', 'data/nets', 'bench'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print('persisting to', DRIVE)

In [ ]:
import os
REPO_URL = 'https://github.com/<you>/chesshack.git'   # <-- set this (or upload to Drive)
if not os.path.isdir('/content/chesshack/.git'):
    !git clone $REPO_URL /content/chesshack
else:
    !cd /content/chesshack && git pull
%cd /content/chesshack
!git log --oneline -1

In [ ]:
# deps: torch ships with Colab (CUDA). add python-chess; fetch Stockfish into tools/
!pip -q install python-chess
import os, urllib.request, tarfile, shutil, glob
os.makedirs('tools', exist_ok=True)
if not os.path.exists('tools/stockfish'):
    url='https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar'
    urllib.request.urlretrieve(url, '/tmp/sf.tar'); tarfile.open('/tmp/sf.tar').extractall('/tmp')
    b=[f for f in glob.glob('/tmp/stockfish/*') if os.path.isfile(f) and os.access(f, os.X_OK)][0]
    shutil.copy(b, 'tools/stockfish'); os.chmod('tools/stockfish', 0o755)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# link data/ + bench/ to Drive so checkpoints + shards + elo history survive restarts.
import os, shutil
for d in ('data', 'bench'):
    local=f'/content/chesshack/{d}'; target=f'{DRIVE}/{d}'
    if os.path.islink(local): continue
    if os.path.isdir(local):
        shutil.copytree(local, target, dirs_exist_ok=True); shutil.rmtree(local, ignore_errors=True)
    os.makedirs(target, exist_ok=True); os.symlink(target, local)
print('Put your locally-labeled shards in', f'{DRIVE}/data/distill/', '(manifest.json + shard_*.npz)')
!ls -la data/distill | head

## Phase 1 — distillation training (GPU)
Trains the production net (C256/B20, ~24.5M) on the Drive shards. Re-run to continue; bench appends to `bench/elo_history.json`.

In [ ]:
!python -m trainer.train --mode distill --data data/distill --net prod --steps 40000 --batch 1024 --lr 2e-3 --workers 8 --out data/nets/distilled.pt

In [ ]:
# Measure Elo vs the Stockfish UCI_Elo ladder (the headline number).
!python -m bench.elo --ckpt data/nets/distilled.pt --games 40 --sims 800

In [ ]:
# Anti-stall sanity before Phase 2: search must beat the raw net.
!python -m bench.search_gain --ckpt data/nets/distilled.pt --sims 600 --games 40

## Phase 2 — self-play (uncapped) — run after Phase 1 lands ~2000+
`trainer/selfplay.py` + `engine/inference_server.py` + `trainer/gate.py` drive the ratchet from the distilled net. (Built in the next milestone.)